## Übung: Streaming

In diesem Assignment wird eine Stream Umgebung simuliert. Es werden synthetisch Twitter Daten generiert und diese in das Verzeichnis `data/` geladen.

Die Twitterdaten sollen anschließend mit Spark Structured Streaming eingelesen und analysiert werden.

Die Datengenerierung erfolgt über ein Python Skript, dass in einem zweiten Container läuft. 

### Vorbereitung der Daten
Überprüfen Sie, ob beim Start von docker compose das Verzeichnis `data` mit Daten befüllt wird. 

### Benötigte Imports laden
Fügen Sie hier alle benötigten Imports ein.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

### Daten Stream erstellen
- Erstellen Sie eine `SparkSession`.
- Legen Sie das folgende Schema an:
  - id, integer
  - text, string
  - timestamp, timestamp
- Erstellen Sie einen Streaming `DataFrame`. Sie können dazu das Beispiel in der [Dokumentation](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#creating-streaming-dataframes-and-streaming-datasets) anpassen.

- Das zu überwachende Directory ist `data`. Das einzulesende Dateiformat ist `csv`. Die Option `header` sollte auf `True` gesetzt werden.

In [2]:
spark = SparkSession.builder.appName("Praktikum2_HashtagCount").config("spark.sql.session.timeZone", "Europe/Berlin").getOrCreate()

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

tweet_df = spark.readStream.format("csv").option("path", "../data/").option("header", "true").schema(schema).load()
tweet_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



### Bearbeitung des Datastroms
Auf dem erstellten streaming DataFrame können verschiedene Operationen durchgeführt werden. Die meisten kennen Sie schon aus der DataFrame API. Für die Analyse der Streamingdaten sollen Sie die folgenden Operationen auf den eingelesenen Dateien ausführen:

- Zerlegen Sie den Text in einzelne Tokens
- Filtern Sie Hashtags
- Zählen Sie für jedes HashTag die Häufigkeit

Die Auswertung soll alle 30 Sekunden die Zusammenfassung der letzten 2 Minuten ausgeben. Dabei soll nach der Event-Zeit (timestamp) ausgewertet werden. Es sollen auch Events berücksichtigt werden, die bis zu einer Minute zu spät eingelaufen sind. Ergänzen Sie das folgende Statement:

Tipp: In der Dokumentation ist das Windowing gut beschrieben.
[Dokumentation](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time)

In [3]:
# Ersetze Sonderzeichen wie "." am Ende des Satzes
tweet_df = tweet_df.withColumn(
    'cleaned_text', 
    regexp_replace(col('text'), r"[^a-zA-Z0-9\s#]", "")
)
    
tweet_df = tweet_df.withColumn("timestamp", col("timestamp") + expr("INTERVAL 2 HOURS")) # Korrektur
tweet_df = tweet_df.withWatermark("timestamp", "1 minute")


hashtags = tweet_df.withColumn('hashtag', explode(split(col('cleaned_text'), ' '))) \
  .filter(col('hashtag').contains('#')) \
  .withColumn('hashtag', lower(trim(col('hashtag')))) \
  .groupBy(window(col("timestamp"), "30 seconds", "10 seconds"), col("hashtag")) \
  .count() \
  .filter(col("count") >= 20)


windowSpec = Window.partitionBy("window").orderBy(col("count").desc())

### Tweets Analyse ausgeben (1)

Schreiben Sie die Tweets in den Hauptspeicher. Mehr Informationen finden Sie [hier](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#starting-streaming-queries).

In [4]:
streamwriter = hashtags.writeStream.outputMode("update").format("memory").trigger(processingTime='10 seconds').queryName("tweets2ram").start()

file_stream = hashtags.writeStream.outputMode("append").format("csv").option("path", "../output/relevant_hashtags/").option("checkpointLocation", "../checkpoints/hashtag_storage/").option("header", "true").trigger(processingTime='10 seconds').start()


print("Stream is running")

Stream is running


Geben Sie alle 10 Sekunden die Liste mit den Hashtags aus dem Hauptspeicher aus. Formulieren Sie dazu im Parameter von `spark.sql` eine SQL Abfrage.

In [5]:
from IPython.display import display, clear_output
import time

while (True):
    clear_output(wait=True)
    display(spark.sql('SELECT * FROM tweets2ram order by window desc, count desc limit 10').show(truncate=False))
    time.sleep(10)


+------------------------------------------+---------+-----+
|window                                    |hashtag  |count|
+------------------------------------------+---------+-----+
|{2026-05-13 18:23:50, 2026-05-13 18:24:20}|#beans   |31   |
|{2026-05-13 18:23:50, 2026-05-13 18:24:20}|#danish  |27   |
|{2026-05-13 18:23:50, 2026-05-13 18:24:20}|#ice     |25   |
|{2026-05-13 18:23:50, 2026-05-13 18:24:20}|#lollipop|23   |
|{2026-05-13 18:23:40, 2026-05-13 18:24:10}|#beans   |35   |
|{2026-05-13 18:23:40, 2026-05-13 18:24:10}|#danish  |27   |
|{2026-05-13 18:23:40, 2026-05-13 18:24:10}|#lollipop|27   |
|{2026-05-13 18:23:40, 2026-05-13 18:24:10}|#ice     |27   |
|{2026-05-13 18:23:30, 2026-05-13 18:24:00}|#beans   |35   |
|{2026-05-13 18:23:30, 2026-05-13 18:24:00}|#lollipop|27   |
+------------------------------------------+---------+-----+



None

KeyboardInterrupt: 

### Stoppen des Streams

In [7]:
streamwriter.stop()

In [8]:
!rm -r data/

rm: cannot remove 'data/': No such file or directory


In [9]:
p.kill()

NameError: name 'p' is not defined